In [ ]:
"""import requests

# Replace the URL with one from the Available Endpoints
url = "https://api.sectors.app/v2/industries/"
api_key = "ff754b804e36ad701c40b38b48364d503da4b6d03eef8fa10b4dd3d2578cefc0"
headers = {"Authorization": api_key}

try:
    response = requests.get(url, headers=headers)
    response.raise_for_status()
    data = response.json()
except requests.exceptions.HTTPError as err:
    raise SystemExit(err)"""

In [ ]:
"""import json

# Asumsi 'data_api' adalah variabel yang berisi response JSON dari Sectors API tadi
data_api = data

# 1. Menyimpan data ke dalam file 'berita_pasar.json'
with open('berita_pasar.json', 'w') as file:
    json.dump(data_api, file, indent=4)

print("Data berhasil disimpan ke berita_pasar.json!")

# 2. Cara membukanya kembali nanti:
with open('berita_pasar.json', 'r') as file:
    data_tersimpan = json.load(file)"""

Data berhasil disimpan ke berita_pasar.json!


In [33]:
import pandas as pd

data_sectors = pd.read_csv('sectors and subsectors.csv')
data_industri = pd.read_csv('industries.csv')
data_subindustri = pd.read_csv('subindustries.csv')

gabungan = pd.merge(data_sectors, data_industri, on='subsector',how='left')
gabungan = pd.merge(gabungan, data_subindustri, on='industry',how='left')
#print(gabungan.columns)

gabungan = gabungan.drop(columns=['Unnamed: 0_x', 'Unnamed: 0_y', 'Unnamed: 0'])

multiple_industry_sectors = gabungan.groupby(['sector', 'subsector', 'industry'])['sub_industry'].value_counts()

gabungan.to_csv('Sector_SubSector_Industry_SubIndustry.csv')


In [35]:
transport_logistic_company = pd.read_csv("Indonesia Stock Exchange Transportation and Logistics Tickers by Industry - Table 1.csv")
transport_logistic_company

,Ticker,Industry Name,Company Name,Sub-sector,Market Capitalization
0,GIAA,Airlines,PT Garuda Indonesia (Persero) Tbk Class B,Airlines / Code: K111,27.28 T IDR
1,JSMR,Transportation,PT Jasa Marga (Persero) Tbk Class B,Not in source,21.7 T IDR
2,TCPI,Transportation,PT Transcoal Pacific,Not in source,11.3 T IDR
3,CMNP,Transportation,PT Citra Marga Nusaphala Persada Tbk,Not in source,9.54 T IDR
4,RMKE,Transportation,PT RMK Energy Tbk,Not in source,9.23 T IDR
...,...,...,...,...,...
63,MIRA,Logistics & Deliveries,PT Mitra International Resources Tbk,Logistics & Deliveries / Code: K211,Not in source
64,PJHB,Logistics & Deliveries,PT Pelayaran Jaya Hidup Baru Tbk,Logistics & Deliveries / Code: K211,Not in source
65,RCCC,Logistics & Deliveries,PT Utama Radar Cahaya Tbk,Logistics & Deliveries / Code: K211,Not in source
66,WBSA,Logistics & Deliveries,PT BSA Logistics Indonesia Tbk,Logistics & Deliveries / Code: K211,Not in source


In [14]:
pd.set_option('display.max_rows', None)

In [2]:
import pandas as pd
df = pd.read_csv('../../dataset/csv/Top 10 transportation companies by market cap.csv')

In [21]:
import requests


def sectors_requester(endpoint, params=None):
  headers = {"Authorization": "8f78cc2a4fa85eb0606e28cf870f93d855d5f8a9b71afa51c2eec738c5147369"}
  url = "https://api.sectors.app/v2/"

  base_url = url + endpoint + "/"
  response = requests.get(base_url, headers=headers, params=params)
  response.raise_for_status()
  data = response.json()

  return data

In [10]:
#pulling corporate action data out of the api
import time
def getting_companies_corporate_actions(ticker_list):
    response_list = []
    for symbol in ticker_list:
        endpoint = 'company/corporate-actions'
        endpoint = endpoint + f'/{symbol}'
        response = sectors_requester(endpoint)
        response_list.append(response)
        time.sleep(10)

    return response_list

In [ ]:
response = getting_companies_corporate_actions(df['symbol'])

In [9]:
#turning corporate action result to dataframe
import pandas as pd
def scraping_corporate_actions_data(response_list):
    event_types = list(response_list[0]['corporate_actions'])
    tables = {}
    for et in event_types:
        rows = []
        for entry in response_list:
            symbol = entry["symbol"]
            records = entry["corporate_actions"].get(et)
            if not records:
                continue
            for r in records:
                rows.append({"symbol": symbol, **r})
        tables[et] = pd.DataFrame(rows)

    long_df = pd.concat([df.assign(event_type=et) for et, df in tables.items() if not df.empty],axis=0, ignore_index=True)
    # 3. put the classifier columns up front for readability
    cols = ["symbol", "event_type"] + [c for c in long_df.columns if c not in ("symbol", "event_type")]
    long_df = long_df[cols]
    return long_df

In [ ]:
df_final = scraping_corporate_actions_data(response)
df_final.to_csv('../../dataset/top10-transportation-by-market-cap-corporate-action.csv', index=False)

In [ ]:
def companies_with_revenue_segments():
    endpoint = 'companies/list_companies_with_segments/'
    response = sectors_requester(endpoint)
    return response

In [177]:
companies_with_revenue_segments_list = companies_with_revenue_segments()

In [217]:
import time
def get_companies_revenue_segments(companies_with_revenue_segments_response, targeted_companies_ticker_list):
    list_of_response = []
    list_of_companies = list(companies_with_revenue_segments_response)

    for ticker in targeted_companies_ticker_list:
        if ticker in list_of_companies:
            endpoint = f'company/get-segments/{ticker}'
            list_of_financial_year = companies_with_revenue_segments_response[ticker]['financial_year']
            for financial_year in list_of_financial_year:
                params = {'financial_year' : financial_year}
                response = sectors_requester(endpoint, params)
                list_of_response.append(response)
                time.sleep(10)
    return list_of_response

In [218]:
response = get_companies_revenue_segments(companies_with_revenue_segments_list,df['symbol'])

In [ ]:
import pandas as pd
def scrapping_company_revenue_segment(company_revenue_segment_response):
    list_of_dataframe = []
    for industri in company_revenue_segment_response:
        ticker = industri['symbol']
        revenue_breakdown = pd.DataFrame(columns=['company_name','value', 'source', 'target'])
        company_name = []
        value = []
        source = []
        target = []
        for i in industri['revenue_breakdown'] : 
            company_name.append(ticker)
            value.append(i['value'])
            source.append(i['source'])
            target.append(i['target'])
        revenue_breakdown['company_name'] = company_name
        revenue_breakdown['value'] = value
        revenue_breakdown['source'] = source
        revenue_breakdown['target'] = target
        revenue_breakdown['financial_year'] = industri['financial_year']

        list_of_dataframe.append(revenue_breakdown)

    final_df = pd.concat(list_of_dataframe,axis=0)
    return final_df

In [ ]:
final_df = scrapping_company_revenue_segment(response)
final_df.to_csv('../../dataset/Sector_SubSector_Industry_SubIndustry.csv',index=False)

In [ ]:
#pulling shareholder composition data out of the api
import time
def get_shareholder_composition(ticker_list):
    response_list = []
    for symbol in ticker_list:
        endpoint = 'company/shareholders-composition'
        endpoint = endpoint + f'/{symbol}'
        current_year = time.localtime().tm_year
        for financial_year in [current_year-1, current_year]:
            params = {'year' : financial_year}
            response = sectors_requester(endpoint, params)
            response_list.append(response)
            time.sleep(10)

    return response_list


In [13]:
response_list = get_shareholder_composition(df['symbol'])

2025
2026
2025
2026
2025
2026
2025
2026
2025
2026
2025
2026
2025
2026
2025
2026
2025
2026
2025
2026


In [14]:
#company shareholder composition
def scraping_shareholder_composition(response_list):
    list_of_dataframe = []
    previous_ticker = ''
    for i in range(len(response_list)):
        symbol = response_list[i]['symbol']
        dataframe = pd.DataFrame(response_list[i]['data'])
        dataframe['symbol'] = symbol
        if symbol == previous_ticker:
            dataframe = pd.concat([list_of_dataframe[len(list_of_dataframe)-1], dataframe], axis=0, ignore_index=True)
            list_of_dataframe[len(list_of_dataframe)-1] = dataframe
        else:
            previous_ticker = symbol
            previous_dataframe = dataframe
            list_of_dataframe.append(dataframe)
    df_final = pd.concat(list_of_dataframe, axis=0, ignore_index=True)
    return df_final


In [15]:
df_final = scraping_shareholder_composition(response_list)
df_final.to_csv('../../dataset/csv/top10-transportation-by-market-cap-shareholder-composition.csv',index=False)

In [4]:
#fixing the corporate action csv
df = pd.read_csv('../../dataset/csv/top10-transportation-by-market-cap-corporate-action.csv')

In [71]:
#turning corporate action result to dataframe
import pandas as pd
def scraping_corporate_actions_data(response_list):
    event_types = list(response_list[0]['corporate_actions'])
    tables = {}
    for et in event_types:
        rows = []
        for entry in response_list:
            symbol = entry["symbol"]
            records = entry["corporate_actions"].get(et)
            if not records:
                continue
            for r in records:
                rows.append({"symbol": symbol, **r})
        if not pd.DataFrame(rows).empty:
            tables[et] = pd.DataFrame(rows)

    return tables

In [88]:
dataframe_json = scraping_corporate_actions_data(response)

In [78]:
import json
def saving_json_dataframe(filename, json_dataframe):
    json_dataframe = {key: dataframe.to_dict(orient='records') for key, dataframe in json_dataframe.items()}
    with open(filename, 'w') as f:
        json.dump(json_dataframe, f, indent=2, default=str)

In [91]:
filename = '../../dataset/json/top10-transportation-by-market-cap-corporate-action.json'
saving_json_dataframe(filename, dataframe_json)

In [92]:
import pandas as pd
import json 
def json_dataframe_to_dataframe(filename):
    with open(filename) as f:
        dataframe_json = json.load(f)

    dataframe_json = {key: pd.DataFrame(records) for key, records in dataframe_json.items()}
    return dataframe_json

In [93]:
dataframe_json = json_dataframe_to_dataframe(filename)

In [1]:
response = {
  "symbol": "BBCA.JK",
  "company_name": "PT Bank Central Asia Tbk.",
  "overview": {
    "listing_board": "Main",
    "industry": "Banks",
    "sub_industry": "Banks",
    "sector": "Financials",
    "sub_sector": "Banks",
    "market_cap": 753611199412500,
    "market_cap_rank": 1,
    "address": "Menara BCA, Grand Indonesia\r\nJalan MH Thamrin No. 1\r\nJakarta 10310",
    "employee_num": 27937,
    "employee_num_rank": 15,
    "listing_date": "2000-05-31",
    "website": "www.bca.co.id",
    "phone": "021-23588000",
    "email": "investor_relations@bca.co.id",
    "last_close_price": 6175,
    "latest_close_date": "2026-07-08",
    "daily_close_change": -0.0198412698412698,
    "all_time_price": {
      "ytd_low": {
        "2026-06-09": 4820
      },
      "52_w_low": {
        "2026-06-09": 4820
      },
      "90_d_low": {
        "2026-06-09": 4820
      },
      "ytd_high": {
        "2026-01-06": 8175
      },
      "52_w_high": {
        "2025-08-13": 8975
      },
      "90_d_high": {
        "2026-04-14": 6800
      },
      "all_time_low": {
        "2004-06-08": 175
      },
      "all_time_high": {
        "2024-09-23": 10950
      }
    },
    "esg_score": 21.44,
    "tags": [
      "dividend-yield-ttm-above-5-percent",
      "esg-under-25",
      "top-90d-transaction-value",
      "top-90d-transaction-volume"
    ],
    "indices": [
      "IDXESGL",
      "ECONOMIC30",
      "IDXG30",
      "IDX30",
      "LQ45",
      "FTSE",
      "SRIKEHATI",
      "KOMPAS100",
      "IDXHIDIV20",
      "IDXQ30"
    ],
    "affiliates": [
      "Djarum",
      "Hartono"
    ]
  },
  "valuation": {
    "last_close_price": 6175,
    "latest_close_date": "2026-07-08",
    "daily_close_change": -0.0198412698412698,
    "forward_pe": 12.7984576757419,
    "intrinsic_value": 13694,
    "historical_valuation": [
      {
        "pb": 4.71766971488811,
        "pe": 25.6154045827394,
        "ps": 11.9285086042203,
        "pcf": 30.8908493442671,
        "peg": 0.8642744448236932,
        "year": 2022,
        "pb_peer_avg": 0.975919472508022,
        "pe_peer_avg": 15.1292552726864,
        "ps_peer_avg": 4.41860675563346,
        "enterprise_to_ebitda": None,
        "enterprise_to_revenue": None
      }
    ]
  },
  "future": {
    "company_value_forecasts": [
      {
        "eps_estimate": 518.5,
        "estimate_year": 2026,
        "revenue_estimate": 126170000000000
      }
    ],
    "company_growth_forecasts": [
      {
        "base_year": 2025,
        "eps_growth": 0.099789989937308,
        "estimate_year": 2026,
        "revenue_growth": 0.12645423259397
      }
    ],
    "analyst_rating_breakdown": {
      "buy": 1,
      "hold": 1,
      "sell": 0,
      "n_analyst": 23,
      "strong_buy": 21,
      "updated_on": "2026-05-07 18:03:41",
      "strong_sell": 0
    }
  },
  "financials": {
    "eps": 471.4536454633092,
    "historical_eps": {
      "2025": {
        "eps": 471.4536454633092,
        "eps_growth": 0.0493
      }
    },
    "historical_financials": [
      {
        "tax": 6854404000000,
        "ebit": None,
        "year": 2018,
        "ebitda": None,
        "revenue": 63028090000000,
        "earnings": 25855154000000,
        "net_debt": 9375812000000,
        "net_loan": None,
        "cash_only": None,
        "provision": None,
        "credit_rwa": None,
        "gross_loan": None,
        "market_rwa": None,
        "total_debt": 9375812000000,
        "cash_inflow": None,
        "cash_outflow": None,
        "fixed_assets": None,
        "gross_profit": None,
        "time_deposit": None,
        "total_assets": 824787944000000,
        "total_equity": 151753427000000,
        "net_cash_flow": None,
        "operating_pnl": 32512504000000,
        "total_capital": None,
        "total_deposit": None,
        "free_cash_flow": 22115523000000,
        "long_term_debt": None,
        "premium_income": None,
        "prepaid_assets": None,
        "cost_of_revenue": None,
        "current_account": None,
        "interest_income": None,
        "non_loan_assets": None,
        "operational_rwa": None,
        "premium_expense": None,
        "savings_account": None,
        "short_term_debt": None,
        "interest_expense": None,
        "end_cash_position": None,
        "operating_expense": None,
        "retained_earnings": None,
        "total_liabilities": 673034517000000,
        "core_capital_tier1": None,
        "industry_breakdown": None,
        "net_premium_income": None,
        "outstanding_shares": 123275000000,
        "allowance_for_loans": None,
        "current_liabilities": None,
        "earnings_before_tax": 32706064000000,
        "financing_cash_flow": None,
        "investing_cash_flow": None,
        "net_interest_income": None,
        "non_interest_income": None,
        "operating_cash_flow": 24462746000000,
        "cash_and_equivalents": None,
        "non_loan_earning_assets": None,
        "high_quality_liquid_asset": None,
        "total_risk_weighted_asset": None,
        "non_loan_non_earning_assets": None,
        "supplementary_capital_tier2": None,
        "non_operating_income_or_loss": None,
        "total_cash_and_due_from_banks": 65239752000000,
        "interest_expense_non_operating": None,
        "non_interest_bearing_liabilities": None,
        "realized_capital_goods_investment": None,
        "other_interest_bearing_liabilities": None
      }
    ],
    "historical_financial_ratio": [
      {
        "year": "2018",
        "capital": {
          "capital_adequacy_ratio": None
        },
        "leverage": {
          "debt_to_asset_ratio": 0.011367542491624975,
          "debt_to_equity_ratio": 4.43505316687181
        },
        "liquidity": {
          "casa_ratio": None,
          "leverage_ratio": None,
          "loan_to_deposit_ratio": None,
          "liquidity_coverage_ratio": None,
          "operating_cash_flow_margin": 0.38812450131362064
        },
        "efficiency": {
          "total_asset_turnover": 0.07641732697295585
        },
        "profitability": {
          "roa": 0.03134763812697049,
          "roe": 0.17037607987594244,
          "efficiency_ratio": 0.03134763812697049,
          "net_profit_margin": 0.4102163654332536,
          "net_interest_margin": None,
          "cost_to_income_ratio": None,
          "operating_profit_margin": 0.5158414922616249
        }
      }
    ],
    "yoy_quarter_earnings_growth": 0.0387073681660018,
    "yoy_quarter_revenue_growth": 0.0110150546891309
  },
  "dividend": {
    "historical_dividends": {
      "2026": {
        "breakdown": [
          {
            "date": "2026-06-17",
            "total": 20,
            "yield": 0.00323886639676113
          }
        ],
        "total_yield": 0.0487449392712551,
        "total_dividend": 301
      }
    },
    "upcoming_dividends": None,
    "yield_ttm": 0.0576518218623482,
    "dividend_yield_avg": {
      "period": 5,
      "avg_yield": 0.0283050359692425
    },
    "dividend_ttm": 356,
    "payout_ratio": 0.748637602001087,
    "cash_payout_ratio": 0.687782037617804,
    "last_ex_dividend_date": "2026-06-17"
  },
  "management": {
    "key_executives": [
      {
        "name": "Gregory Hendra Lembong",
        "position": "President Director"
      }
    ],
    "executives_shareholdings": [
      {
        "name": "Armand Wahyudi Hartono",
        "position": "Vice President Director",
        "share_amount": 4256065,
        "share_percentage": 0.00003
      }
    ]
  },
  "ownership": {
    "major_shareholders": [
      {
        "name": "PT Dwimuria Investama Andalan",
        "share_value": 418232441250000,
        "share_amount": 67729950000,
        "share_percentage": "0.54942"
      }
    ],
    "top_transactions": {
      "date": "2026-04-30",
      "top_buyers": [
        {
          "name": "Fidelity Institutional Asset Management",
          "changeAmount": 186742480
        }
      ],
      "top_sellers": [
        {
          "name": "Fidelity Management & Research Company LLC",
          "changeAmount": -833622654
        }
      ]
    },
    "institutional_transaction_flow": [
      {
        "date": "2026-04-30",
        "net_transaction": -4575387238
      }
    ],
    "whale_investors": [
      "Anthoni Salim"
    ],
    "conglomerates_group": [
      "Djarum Group"
    ]
  },
  "peers": [
    {
      "peers_data": {
        "companies": [
          {
            "year": 2025,
            "group": [
              "self"
            ],
            "pb_mrq": 2.96707963218973,
            "pe_ttm": 13.2251882418153,
            "symbol": "BBCA.JK",
            "market_cap": 768866486850000,
            "net_income": 57537287000000,
            "company_name": "PT Bank Central Asia Tbk.",
            "employee_num": 27937,
            "total_assets": 1586830000000000,
            "total_equity": 281687555000000,
            "pretax_income": 71260876000000,
            "total_revenue": 112006326000000,
            "point_summaries": [
              {
                "name": "value",
                "point": 10.5,
                "maxpoint": 18
              }
            ],
            "yearly_mcap_chg": -0.275862068965517,
            "operating_expense": 36734403000000,
            "revenue_breakdown": None,
            "total_liabilities": 1305140000000000,
            "int_income_breakdown": [
              {
                "class": "Loans & Deposits",
                "amount": 67446394000000,
                "category": "Loans"
              }
            ],
            "operating_expense_breakdown": [
              {
                "class": "Salaries & Benefits",
                "amount": 17780770000000,
                "category": "Personnel expenses"
              }
            ]
          }
        ],
        "group_name": {
          "sector": "Financials",
          "industry": "Banks",
          "sub_sector": "Banks",
          "sub_industry": "Banks"
        }
      }
    }
  ]
}

In [27]:
#pulling company report data out of the api
import time
def company_report(ticker_list):
    response_list = []
    for symbol in ticker_list:
        endpoint = 'company/report'
        endpoint = endpoint + f'/{symbol}'
        params = {
            'sections': 'dividend,financials,future,management,overview,ownership,peers,valuation'
        }
        current_year = time.localtime().tm_year
        response = sectors_requester(endpoint, params)
        response_list.append(response)
        time.sleep(10)

    return response_list


In [48]:
complete_response_list = company_report(df['symbol'][1:].to_list())

In [51]:
complete_response_list

[{'symbol': 'TMAS.JK',
  'company_name': 'PT Temas Tbk',
  'dividend': {'historical_dividends': {'2020': {'breakdown': [{'date': '2020-07-06',
       'total': 3.6,
       'yield': 0.34527599811554}],
     'total_yield': 0.34527599811554,
     'total_dividend': 3.6},
    '2021': {'breakdown': [{'date': '2021-06-22',
       'total': 43.82,
       'yield': 1.24202001094818}],
     'total_yield': 1.24202001094818,
     'total_dividend': 43.82},
    '2022': {'breakdown': [{'date': '2022-12-22',
       'total': 52.28,
       'yield': 0.253684014081955},
      {'date': '2022-06-20', 'total': 65.79, 'yield': 0.319240003824234},
      {'date': '2022-01-07', 'total': 21.85, 'yield': 0.106025002896786}],
     'total_yield': 0.678949020802975,
     'total_dividend': 139.92},
    '2023': {'breakdown': [{'date': '2023-04-28',
       'total': 80,
       'yield': 0.0343975983560085}],
     'total_yield': 0.0343975983560085,
     'total_dividend': 80},
    '2024': {'breakdown': [{'date': '2024-04-16',


In [59]:
complete_response_list = response_list + complete_response_list

In [66]:
complete_response_list

[{'symbol': 'GIAA.JK',
  'company_name': 'Garuda Indonesia (Persero) Tbk',
  'dividend': {'historical_dividends': None,
   'upcoming_dividends': None,
   'yield_ttm': None,
   'dividend_yield_avg': None,
   'dividend_ttm': None,
   'payout_ratio': None,
   'cash_payout_ratio': None,
   'last_ex_dividend_date': None},
  'financials': {'eps': -13.258195149083868,
   'historical_eps': {'2020': {'eps': -1221.9310590519979},
    '2021': {'eps': -2108.050607478869, 'eps_growth': 0.7252},
    '2022': {'eps': 637.0918791834393, 'eps_growth': -1.3022},
    '2023': {'eps': 42.199814044767805, 'eps_growth': -0.9338},
    '2024': {'eps': -12.841675971460774, 'eps_growth': -1.3043},
    '2025': {'eps': -13.258195149083868, 'eps_growth': 0.0324}},
   'historical_financials': [{'tax': 635855518124,
     'ebit': 2668917605626,
     'year': 2019,
     'ebitda': 2938370549920,
     'revenue': 63479646152069,
     'earnings': 96985085441,
     'net_debt': 20602126749464,
     'cash_only': None,
     'tot

In [61]:
from __future__ import annotations
import pandas as pd


def _get(d: dict, *path, default=None):
    """Safely walk a nested dict, returning `default` if anything is missing."""
    cur = d
    for key in path:
        if not isinstance(cur, dict) or key not in cur:
            return default
        cur = cur[key]
    return cur if cur is not None else default


def company_report_dataframe(data_list: list[dict]) -> dict[str, pd.DataFrame]:
    """
    Parameters
    ----------
    data_list : list[dict]
        A list of stock-profile JSON objects, each shaped like the
        BBCA.JK example (top-level keys: symbol, company_name, overview,
        valuation, future, financials, dividend, management, ownership,
        peers, ...).

    Returns
    -------
    dict[str, pd.DataFrame]
        A dictionary of tidy, analysis-ready DataFrames (see module docstring
        for the list of keys).
    """

    overview_rows = []
    valuation_hist_rows = []
    financials_hist_rows = []
    ratio_rows = []
    dividend_rows = []
    dividend_breakdown_rows = []
    forecast_rows = []
    exec_rows = []
    exec_shareholding_rows = []
    shareholder_rows = []
    peer_rows = []

    for entry in data_list:
        symbol = entry.get("symbol")
        company_name = entry.get("company_name")
        key = {"symbol": symbol, "company_name": company_name}

        # ---------- overview (1 row per stock) ----------
        ov = entry.get("overview", {}) or {}
        atp = ov.get("all_time_price", {}) or {}

        def _pt_val(band):
            band = atp.get(band, {}) or {}
            if not band:
                return None, None
            date, val = next(iter(band.items()))
            return date, val

        row = {
            **key,
            "sector": ov.get("sector"),
            "industry": ov.get("industry"),
            "sub_sector": ov.get("sub_sector"),
            "listing_board": ov.get("listing_board"),
            "listing_date": ov.get("listing_date"),
            "market_cap": ov.get("market_cap"),
            "market_cap_rank": ov.get("market_cap_rank"),
            "employee_num": ov.get("employee_num"),
            "esg_score": ov.get("esg_score"),
            "last_close_price": ov.get("last_close_price"),
            "latest_close_date": ov.get("latest_close_date"),
            "daily_close_change": ov.get("daily_close_change"),
            "website": ov.get("website"),
            "tags": ", ".join(ov.get("tags") or []),
            "indices": ", ".join(ov.get("indices") or []),
            "affiliates": ", ".join(ov.get("affiliates") or []),
            "intrinsic_value": _get(entry, "valuation", "intrinsic_value"),
            "forward_pe": _get(entry, "valuation", "forward_pe"),
        }
        for band in ["ytd_low", "ytd_high", "52_w_low", "52_w_high",
                     "90_d_low", "90_d_high", "all_time_low", "all_time_high"]:
            date, val = _pt_val(band)
            row[f"{band}_date"] = date
            row[f"{band}_price"] = val
        overview_rows.append(row)

        # ---------- valuation history (1 row per stock per year) ----------
        for v in _get(entry, "valuation", "historical_valuation", default=[]):
            valuation_hist_rows.append({**key, **v})

        # ---------- financials history (1 row per stock per year) ----------
        for fin in _get(entry, "financials", "historical_financials", default=[]):
            financials_hist_rows.append({**key, **fin})

        # ---------- financial ratios (nested dict -> flat row) ----------
        for r in _get(entry, "financials", "historical_financial_ratio", default=[]):
            flat = {**key, "year": r.get("year")}
            for group, metrics in r.items():
                if group == "year" or not isinstance(metrics, dict):
                    continue
                for metric_name, metric_val in metrics.items():
                    flat[f"{group}_{metric_name}"] = metric_val
            ratio_rows.append(flat)

        # ---------- dividends (1 summary row per stock) ----------
        div = entry.get("dividend", {}) or {}
        dividend_rows.append({
            **key,
            "yield_ttm": div.get("yield_ttm"),
            "dividend_ttm": div.get("dividend_ttm"),
            "payout_ratio": div.get("payout_ratio"),
            "cash_payout_ratio": div.get("cash_payout_ratio"),
            "last_ex_dividend_date": div.get("last_ex_dividend_date"),
            "avg_yield_5y": _get(div, "dividend_yield_avg", "avg_yield"),
        })
        for year, ydata in (div.get("historical_dividends") or {}).items():
            for pay in ydata.get("breakdown", []):
                dividend_breakdown_rows.append({
                    **key, "year": year,
                    "date": pay.get("date"),
                    "amount": pay.get("total"),
                    "yield": pay.get("yield"),
                })

        # ---------- forecasts / analyst ratings (1 row per stock) ----------
        val_fc = (_get(entry, "future", "company_value_forecasts", default=[]) or [{}])[0]
        growth_fc = (_get(entry, "future", "company_growth_forecasts", default=[]) or [{}])[0]
        ratings = _get(entry, "future", "analyst_rating_breakdown", default={}) or {}
        forecast_rows.append({
            **key,
            **{f"forecast_{k}": v for k, v in val_fc.items()},
            **{f"growth_{k}": v for k, v in growth_fc.items()},
            **{f"rating_{k}": v for k, v in ratings.items()},
        })

        # ---------- management ----------
        for e in _get(entry, "management", "key_executives", default=[]):
            exec_rows.append({**key, **e})
        for e in _get(entry, "management", "executives_shareholdings", default=[]):
            exec_shareholding_rows.append({**key, **e})

        # ---------- ownership ----------
        for s in _get(entry, "ownership", "major_shareholders", default=[]):
            shareholder_rows.append({**key, **s})

        # ---------- peers ----------
        for peer_block in entry.get("peers", []) or []:
            companies = _get(peer_block, "peers_data", "companies", default=[])
            group_name = _get(peer_block, "peers_data", "group_name", default={}) or {}
            for c in companies:
                c = dict(c)
                c.pop("point_summaries", None)
                c.pop("int_income_breakdown", None)
                c.pop("operating_expense_breakdown", None)
                peer_rows.append({**key, **group_name, **c})

    return {
        "overview": pd.DataFrame(overview_rows),
        "valuation_history": pd.DataFrame(valuation_hist_rows),
        "financials_history": pd.DataFrame(financials_hist_rows),
        "financial_ratios": pd.DataFrame(ratio_rows),
        "dividends": pd.DataFrame(dividend_rows),
        "dividend_breakdown": pd.DataFrame(dividend_breakdown_rows),
        "forecasts": pd.DataFrame(forecast_rows),
        "executives": pd.DataFrame(exec_rows),
        "executive_shareholdings": pd.DataFrame(exec_shareholding_rows),
        "major_shareholders": pd.DataFrame(shareholder_rows),
        "peers": pd.DataFrame(peer_rows),
    }

json_dataframe = company_report_dataframe(complete_response_list)

# json_dataframe["overview"]                 # 1 row per stock
# json_dataframe["valuation_history"]        # 1 row per stock per year
# json_dataframe["financials_history"]       # 1 row per stock per year
# json_dataframe["financial_ratios"]         # 1 row per stock per year (flattened)
# json_dataframe["dividends"]                # 1 row per stock
# json_dataframe["dividend_breakdown"]       # 1 row per stock per payment date
# json_dataframe["forecasts"]                # 1 row per stock (value + growth + ratings)
# json_dataframe["executives"]               # 1 row per exec
# json_dataframe["executive_shareholdings"]  # 1 row per exec shareholding
# json_dataframe["major_shareholders"]       # 1 row per major shareholder
# json_dataframe["peers"]                    # 1 row per peer company

In [88]:
json_dataframe['overview'].columns

Index(['symbol', 'company_name', 'sector', 'industry', 'sub_sector',
       'listing_board', 'listing_date', 'market_cap', 'market_cap_rank',
       'employee_num', 'esg_score', 'last_close_price', 'latest_close_date',
       'daily_close_change', 'website', 'tags', 'indices', 'affiliates',
       'intrinsic_value', 'forward_pe', 'ytd_low_date', 'ytd_low_price',
       'ytd_high_date', 'ytd_high_price', '52_w_low_date', '52_w_low_price',
       '52_w_high_date', '52_w_high_price', '90_d_low_date', '90_d_low_price',
       '90_d_high_date', '90_d_high_price', 'all_time_low_date',
       'all_time_low_price', 'all_time_high_date', 'all_time_high_price'],
      dtype='str')

In [69]:
filename='../../dataset/json/top10-transportation-by-marketcap-company_report.json'

In [82]:
saving_json_dataframe(filename, json_dataframe)